## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [17]:
from functools import wraps


def flow_rate(weight, time):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time

weight = 0.5
time = 3

In [18]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

In [19]:
def flow_rate(weight, time, *, period=60, units_per_kg=1000):
    """Oblicza przepływ w wybranych jednostkach i okresie czasu."""
    return weight / time * period * units_per_kg

In [20]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [21]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [22]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

1e+04 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

Błąd występuje, ponieważ argumenty po * muszą być przekazane z nazwami

In [23]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

True


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [24]:
from time import time

def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        elapsed = time() - start
        print(f'Function {func.__name__} took: {elapsed:.5f} seconds')
        return result

    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.06371 seconds
Function squares_comprehension took: 0.05717 seconds
Function squares_map took: 0.00000 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [25]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h,
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    h = epsilon if epsilon is not None else 1000 * sys.float_info.epsilon

    def decorator(func):
        @wraps(func)
        def derivative(x):
            return (func(x + h) - func(x)) / h

        return derivative

    return decorator


@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
69.57735617809249 69.5673561781065 	 True
3629.713829286629 3629.7127842086484 	 True
1650.7851875037886 1650.7751875095116 	 True
2043794.0597534177 2043794.0397727974 	 True
1212.9469256731682 1212.9369256711625 	 True
1103412.0053052902 1103411.9892424583 	 True
259.91302740221727 259.90302740260887 	 True
50662.19163127243 50662.187739780944 	 True
1534.595045261085 1534.5850452662328 	 True
1766213.4647369382 1766213.4458660744 	 True
1993.545335053932 1993.5353350462926 	 True
2980637.371540069 2980637.3490586006 	 True
1831.6035697702318 1831.5935697710597 	 True
2516051.2804985046 2516051.25362002 	 True
123.07887364581802 123.06887364585073 	 True
11359.462596010415 11359.460745343778 	 True
528.0237261395087 528.0137261396634 	 True
209098.87850284574 209098.87124391858 	 True
3.9818909104211375 3.971890910421161 	 True
11.831997631794875 11.831938053214678 	 True
1113.0326531012543 1113.0226530975783 	 True
929114.5831346511 929114.569731279 	 True
628.54

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [26]:
from functools import wraps
from inspect import signature

def accepts(*accepetedTypes):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane
       w argumentach dekoratora"""
    def decorator(func):
        function_signature = signature(func)
        parameter_names = tuple(function_signature.parameters)
        if len(accepetedTypes) != len(parameter_names):
            raise TypeError('Liczba typów musi odpowiadać liczbie parametrów funkcji')

        @wraps(func)
        def wrapper(*args, **kwargs):
            bound = function_signature.bind(*args, **kwargs)
            bound.apply_defaults()
            for name, expected_type in zip(parameter_names, accepetedTypes):
                if not isinstance(bound.arguments[name], expected_type):
                    raise TypeError(f'Argument {name} powinien mieć typ {expected_type.__name__}')
            return func(*args, **kwargs)

        return wrapper

    return decorator


@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [27]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            if len(types) == 1:
                valid = isinstance(result, types[0])
            else:
                valid = (isinstance(result, tuple) and len(result) == len(types) and
                         all(isinstance(value, expected_type)
                             for value, expected_type in zip(result, types)))
            if not valid:
                raise TypeError('Funkcja zwróciła wartość niewłaściwego typu')
            return result

        return wrapper

    return decorator


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [28]:
from functools import wraps
from random import random

def cached(func):
    """Zapamiętuje wyniki funkcji dla wcześniej użytych argumentów."""
    cache = {}
    calls = 0
    evaluations = 0

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal cache, calls, evaluations
        calls += 1
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
            evaluations += 1
        return cache[key]

    def cache_reset():
        cache.clear()

    def cache_status():
        return f'Function {func.__name__} called {calls} times, evaluated {evaluations} times'

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper

@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [29]:
from functools import wraps

def zlozenie(n):
    if n < 0:
        raise ValueError('Liczba złożeń nie może być ujemna')

    def decorator(func):
        @wraps(func)
        def wrapper(value):
            result = value
            for _ in range(n):
                result = func(result)
            return result

        return wrapper

    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [30]:
def my_function(x, y):
    operations = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0,
    }
    if x not in operations:
        raise ValueError(f'Nieobsługiwana wartość: {x}')
    return operations[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
